In [ ]:
import novae

In [ ]:
import scanpy as sc

DATA_PATH="./SEAAD_train.h5ad"
adata_t = sc.read_h5ad(DATA_PATH)

In [ ]:
adata_t

In [ ]:
#sc.pl.spatial(adata_t,color="Subclass")
sc.pl.spatial(adata_t, color='Class', spot_size=10)

In [ ]:
DATA_PATH="./SEAAD_test.h5ad"
adata_test = sc.read_h5ad(DATA_PATH)

In [ ]:
novae.spatial_neighbors(adata_t, radius=10)

In [ ]:
model = novae.Novae.from_pretrained("MICS-Lab/novae-human-0")
model.fine_tune(adata_t,max_epochs=1)
model.compute_representations(adata_t)

In [ ]:
novae.spatial_neighbors(adata_test, radius=10)

In [ ]:
model.fine_tune(adata_test,max_epochs=1)
model.compute_representations(adata_test)

In [ ]:
import numpy as np
np.save("emb_train.npy",adata_t.obsm["novae_latent"])
np.save("emb_test.npy",adata_test.obsm["novae_latent"])

In [ ]:
import numpy as np
adata_t.obsm["novae_latent"]=np.load("emb_train.npy")
adata_test.obsm["novae_latent"]=np.load("emb_test.npy")

In [ ]:
X_train = np.array(adata_t.obsm["novae_latent"])
X_test = np.array(adata_test.obsm["novae_latent"])

In [ ]:
set(adata_test.obs["Subclass"]) - set(adata_t.obs["Subclass"])


In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train = le.fit_transform(adata_t.obs["Subclass"])
y_test = le.transform(adata_test.obs["Subclass"])

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# ==== Convert to tensors ====
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

device =  "cpu"

# ==== Compute class weights ====
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

# ==== Build dataset and balanced sampler ====
train_dataset = TensorDataset(X_train_t, y_train_t)
class_sample_counts = np.bincount(y_train)
weights_per_sample = 1.0 / class_sample_counts[y_train]
sampler = WeightedRandomSampler(weights_per_sample, len(weights_per_sample))

train_loader = DataLoader(train_dataset, batch_size=512, sampler=sampler)

# ==== Define MLP ====
class MLPClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dim=128):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes)
        )
    def forward(self, x):
        return self.fc(x)

input_dim = X_train.shape[1]
num_classes = len(classes)
model = MLPClassifier(input_dim, num_classes).to(device)

# ==== Optimiser and loss ====
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=1e-1)
epochs = 100

# ==== Training loop ====
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}")

print("✅ Training complete!")


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, average_precision_score, classification_report,roc_auc_score

model.eval()
with torch.no_grad():
    probs = torch.softmax(model(X_test_t.to(device)), dim=1).cpu().numpy()
    preds = np.argmax(probs, axis=1)

accuracy = accuracy_score(y_test, preds)
precision = precision_score(y_test, preds, average="macro")
recall = recall_score(y_test, preds, average="macro")
f1 = f1_score(y_test, preds, average="macro")
y_true_bin = np.eye(num_classes)[y_test]
pr_auc = average_precision_score(y_true_bin, probs, average="macro")
roc_auc=roc_auc_score(y_true_bin, probs, average="macro")

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision (macro): {precision:.4f}")
print(f"Recall (macro): {recall:.4f}")
print(f"F1-score (macro): {f1:.4f}")
print(f"PR-AUC (macro): {pr_auc:.4f}")
print(f"ROC-AUC (macro): {roc_auc:.4f}")

print("\nPer-class metrics:")
print(classification_report(y_test, preds, target_names=le.classes_))


In [ ]:
import torch.nn as nn
import torch.optim as optim

class MLPClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dim=128, dropout=0.1):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes)
        )
    def forward(self, x):
        return self.fc(x)

input_dim = X_train.shape[1]
num_classes = len(np.unique(y_train))
model = MLPClassifier(input_dim, num_classes)


In [ ]:
import torch

device =  "cpu"
model.to(device)

X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_t = torch.tensor(y_test, dtype=torch.long).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
epochs = 1000

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    classification_report
)
import numpy as np
import torch

# --- Get predictions and probabilities ---
model.eval()
with torch.no_grad():
    logits = model(X_test_t)
    probs = torch.softmax(logits, dim=1).cpu().numpy()
    pred_labels = np.argmax(probs, axis=1)

# --- Compute metrics ---
accuracy = accuracy_score(y_test, pred_labels)
precision = precision_score(y_test, pred_labels, average="macro")
recall = recall_score(y_test, pred_labels, average="macro")
macro_f1 = f1_score(y_test, pred_labels, average="macro")

# --- Compute macro PR-AUC ---
# Convert true labels to one-hot for multiclass PR-AUC
y_true_bin = np.eye(len(np.unique(y_test)))[y_test]
pr_auc = average_precision_score(y_true_bin, probs, average="macro")

# --- Print results ---
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision (macro): {precision:.4f}")
print(f"Recall (macro): {recall:.4f}")
print(f"F1-score (macro): {macro_f1:.4f}")
print(f"PR-AUC (macro): {pr_auc:.4f}\n")

# --- Per-class breakdown ---
print(classification_report(y_test, pred_labels, target_names=le.classes_))



In [ ]:
data = adata_test.copy()
sc.pp.filter_genes(data, min_cells=3) 

 

sc.pp.highly_variable_genes( data, flavor="cell_ranger", n_top_genes=2000 )# drop batch_key if single batch ) 

data = data[:, data.var['highly_variable']].copy() 

 

# Scale data (optional) 

sc.pp.scale(data) 

 

# Perform PCA (optional, for dimensionality reduction before UMAP) 

sc.tl.pca(data, svd_solver='arpack') 

 

sc.pp.neighbors(data, use_rep='X_pca', random_state=42) 

In [ ]:
sc.tl.umap(data, random_state=42) 
data.obs["Predictions"] = le.inverse_transform(pred_labels)
 

# --- Plot --- 


In [ ]:
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (6, 6) 

fig = sc.pl.umap( data, 
color=['Predictions', 'Subclass'], 
palette='Paired', 
title=['Predictions', 'Ground Truth'], 
show=False, 
return_fig=True ) 

 

# --- Combine legends into one --- 

handles, labels = [], [] 

for ax in fig.axes:
    h, l = ax.get_legend_handles_labels() 
    handles.extend(h) 
    labels.extend(l) 
    ax.get_legend().remove() 

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, pred_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)

fig, ax = plt.subplots(figsize=(10, 10))
disp.plot(ax=ax, xticks_rotation=90, cmap='Blues', colorbar=True)
plt.title("Confusion Matrix – Predicted vs True Cell Types")
plt.show()


In [ ]:
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Compute normalised confusion matrix (per true label)
cm = confusion_matrix(y_test, pred_labels, normalize='true')

plt.figure(figsize=(12, 10))
sns.heatmap(cm, cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_,
            square=True, cbar_kws={'label': 'Proportion'}, linewidths=0.5)
plt.xlabel("Predicted Cell Type")
plt.ylabel("True Cell Type")
plt.title("Normalised Confusion Matrix – Cell Type Annotation")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()
